In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# -------------------------
# 1. Spark Session
# -------------------------
spark = SparkSession.builder \
    .appName("MovieLens Pipeline") \
    .config("spark.cassandra.connection.host", "localhost") \
    .getOrCreate()

sc = spark.sparkContext

# -------------------------
# 2. Load from HDFS
# -------------------------
ratings_rdd = sc.textFile("hdfs:///movielens1/u.data")
users_rdd = sc.textFile("hdfs:///movielens1/u.user")
movies_rdd = sc.textFile("hdfs:///movielens1/u.item")

# -------------------------
# 3. Parse RDDs
# -------------------------
ratings_rdd = ratings_rdd.map(lambda x: x.split("\t"))
users_rdd = users_rdd.map(lambda x: x.split("|"))
movies_rdd = movies_rdd.map(lambda x: x.split("|"))

# -------------------------
# 4. Convert to DataFrames
# -------------------------
ratings_df = ratings_rdd.map(lambda x: (
    int(x[0]), int(x[1]), int(x[2]), int(x[3])
)).toDF(["user_id", "movie_id", "rating", "timestamp"])

users_df = users_rdd.map(lambda x: (
    int(x[0]), int(x[1]), x[2], x[3], x[4]
)).toDF(["user_id", "age", "gender", "occupation", "zipcode"])

movies_df = movies_rdd.map(lambda x: (
    int(x[0]), x[1],
    int(x[6]), int(x[7]), int(x[8]), int(x[9]), int(x[10]),
    int(x[11]), int(x[12]), int(x[13]), int(x[14]), int(x[15]),
    int(x[16]), int(x[17]), int(x[18]), int(x[19]), int(x[20]),
    int(x[21]), int(x[22]), int(x[23])
)).toDF([
    "movie_id", "title",
    "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "FilmNoir",
    "Horror", "Musical", "Mystery", "Romance", "SciFi",
    "Thriller", "War", "Western"
])

# -------------------------
# 5. Temp Views
# -------------------------
ratings_df.createOrReplaceTempView("ratings")
users_df.createOrReplaceTempView("users")
movies_df.createOrReplaceTempView("movies")

# -------------------------
# TASK 1 Average Rating Per Movie
# -------------------------
avg_rating = spark.sql("""
SELECT
    m.movie_id,
    m.title,
    ROUND(AVG(r.rating), 2) AS avg_rating
FROM ratings r
JOIN movies m
ON r.movie_id = m.movie_id
GROUP BY m.movie_id, m.title
ORDER BY avg_rating DESC
""")

print("\nTASK 1: AVERAGE RATING PER MOVIE")
avg_rating.show(10, False)

# -------------------------
# TASK 2 Top 10 Movies
# -------------------------
top10_movies = avg_rating.limit(10)

print("\nTASK 2: TOP 10 MOVIES WITH HIGHEST AVERAGE RATINGS")
top10_movies.show(10, False)

# -------------------------
# TASK 3 Active Users (>=50 Ratings)
# Favourite Genre
# -------------------------
active_users = spark.sql("""
SELECT
    user_id,
    COUNT(movie_id) AS total_ratings
FROM ratings
GROUP BY user_id
HAVING COUNT(movie_id) >= 50
""")

active_ratings = active_users.join(ratings_df, "user_id")
user_movies = active_ratings.join(movies_df, "movie_id")

genres = [
    "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "FilmNoir",
    "Horror", "Musical", "Mystery", "Romance", "SciFi",
    "Thriller", "War", "Western"
]

genre_counts = None

for genre in genres:

    temp = user_movies.filter(col(genre) == 1) \
        .groupBy("user_id") \
        .count() \
        .withColumnRenamed("count", "genre_count") \
        .withColumn("genre", lit(genre))

    if genre_counts is None:
        genre_counts = temp
    else:
        genre_counts = genre_counts.union(temp)

genre_counts.createOrReplaceTempView("genre_counts")

fav_genre = spark.sql("""
SELECT
    user_id,
    genre AS favourite_genre,
    genre_count
FROM
(
    SELECT *,
           ROW_NUMBER() OVER
           (
               PARTITION BY user_id
               ORDER BY genre_count DESC
           ) AS rn
    FROM genre_counts
) t
WHERE rn = 1
ORDER BY user_id
""")

print("\nTASK 3: ACTIVE USERS AND FAVOURITE GENRE")
fav_genre.show(10, False)

# -------------------------
# TASK 4 Users Age < 20
# -------------------------
young_users = spark.sql("""
SELECT *
FROM users
WHERE age < 20
""")

print("\nTASK 4: USERS AGE < 20")
young_users.show()

# -------------------------
# TASK 5 Scientists Age 30-40
# -------------------------
scientists = spark.sql("""
SELECT *
FROM users
WHERE occupation = 'scientist'
AND age BETWEEN 30 AND 40
""")

print("\nTASK 5: SCIENTISTS AGE 30-40")
scientists.show()

# -------------------------
# 6. Write to Cassandra
# -------------------------
avg_rating.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("overwrite") \
    .options(table="avg_ratings", keyspace="movielens3") \
    .save()

top10_movies.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("overwrite") \
    .options(table="top_movies", keyspace="movielens3") \
    .save()

fav_genre.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("overwrite") \
    .options(table="user_fav_genres", keyspace="movielens3") \
    .save()

young_users.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("overwrite") \
    .options(table="young_users", keyspace="movielens3") \
    .save()

scientists.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("overwrite") \
    .options(table="scientist_users", keyspace="movielens3") \
    .save()

# -------------------------
# 7. Read Back from Cassandra
# -------------------------
print("\nREADING FROM CASSANDRA")

check_df = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="ratings", keyspace="movielens3") \
    .load()

check_df.show(5)

spark.stop()